# TIỀN XỬ LÝ VÀ CHUYỂN ĐỔI BỘ DỮ LIỆU KIMORE
Notebook này thực hiện chuyển đổi ma trận tọa độ Kinect v2 (100 cột) và điểm số lâm sàng từ tập dữ liệu KIMORE sang chuẩn 13 khớp giải phẫu tương thích với pipeline đánh giá động tác:
1. **`student_dance_kimore.csv`**: Dữ liệu các bài tập của học viên với nhãn nhị phân (PASS: 1 / FAIL: 0).
2. **`teacher_dance_kimore.csv`**: Dữ liệu bài tập chuẩn của giáo viên (lấy từ mẫu đạt điểm tuyệt đối 50/50, gán nhãn 1).

In [ ]:
import numpy as np
import pandas as pd
import os

# Bảng ánh xạ 13 khớp MediaPipe từ mô hình 25 khớp Kinect v2
KINECT_TO_MEDIAPIPE = {
    'nose': 3,            # Head
    'left_shoulder': 4,   # ShoulderLeft
    'right_shoulder': 8,  # ShoulderRight
    'left_elbow': 5,      # ElbowLeft
    'right_elbow': 9,     # ElbowRight
    'left_wrist': 6,      # WristLeft
    'right_wrist': 10,    # WristRight
    'left_hip': 12,       # HipLeft
    'right_hip': 16,      # HipRight
    'left_knee': 13,      # KneeLeft
    'right_knee': 17,     # KneeRight
    'left_ankle': 14,     # AnkleLeft
    'right_ankle': 18     # AnkleRight
}

INPUT_X_PATH = 'Train_X.csv'
INPUT_Y_PATH = 'Train_Y.csv'
OUTPUT_STUDENT_PATH = 'student_dance_kimore.csv'
OUTPUT_TEACHER_PATH = 'teacher_dance_kimore.csv'
SCORE_THRESHOLD = 35.0  # Điểm >= 35 gán nhãn 1 (PASS), < 35 gán nhãn 0 (FAIL)
FPS = 30.0

## 1. Chuyển đổi dữ liệu học viên (`student_dance_kimore.csv`)

In [ ]:
print("=== 1. CHUYỂN ĐỔI DỮ LIỆU HỌC VIÊN ===")
df_x = pd.read_csv(INPUT_X_PATH, header=None)
df_y = pd.read_csv(INPUT_Y_PATH, header=None)

total_rows_x = len(df_x)
num_samples = len(df_y)
frames_per_sample = total_rows_x // num_samples

print(f"- Tổng số dòng Train_X : {total_rows_x:,}")
print(f"- Tổng số bài tập (Train_Y)  : {num_samples:,}")
print(f"- Số frame mỗi bài tập       : {frames_per_sample} frames")

# Nhị phân hóa nhãn theo ngưỡng điểm lâm sàng
scores = df_y[0].values
binary_labels = (scores >= SCORE_THRESHOLD).astype(int)

student_rows = []
for row_idx in range(total_rows_x):
    sample_id = row_idx // frames_per_sample
    frame_in_sample = row_idx % frames_per_sample
    frame_label = int(binary_labels[sample_id])
    timestamp = round(frame_in_sample / FPS, 4)
    row_x = df_x.iloc[row_idx].values
    
    for joint_name, kinect_idx in KINECT_TO_MEDIAPIPE.items():
        base_col = kinect_idx * 4
        x = float(row_x[base_col])
        y = float(row_x[base_col + 1])
        z = float(row_x[base_col + 2])
        state = float(row_x[base_col + 3])
        
        if state == 2.0:
            visibility = 1.0
        elif state == 1.0:
            visibility = 0.5
        else:
            visibility = 0.0
            x, y, z = np.nan, np.nan, np.nan
            
        student_rows.append({
            'Video_ID': sample_id + 1,
            'Frame': frame_in_sample,
            'Timestamp': timestamp,
            'Joint_Name': joint_name,
            'X': x,
            'Y': y,
            'Z': z,
            'Visibility': visibility,
            'Label': frame_label
        })

df_student = pd.DataFrame(student_rows)
df_student.to_csv(OUTPUT_STUDENT_PATH, index=False)

pass_cnt = sum(binary_labels == 1)
fail_cnt = sum(binary_labels == 0)
print(f"-> Đã lưu file học viên: {OUTPUT_STUDENT_PATH} ({len(df_student):,} dòng)")
print(f"-> Phân bố nhãn: PASS={pass_cnt} ({pass_cnt/num_samples*100:.1f}%), FAIL={fail_cnt} ({fail_cnt/num_samples*100:.1f}%)")

## 2. Tạo dữ liệu chuẩn giáo viên (`teacher_dance_kimore.csv`)

In [ ]:
print("\n=== 2. TẠO DỮ LIỆU CHUẨN GIÁO VIÊN ===")
# Tìm bài tập có điểm số tuyệt đối 50.0
perfect_indices = np.where(scores == 50.0)[0]
if len(perfect_indices) > 0:
    teacher_sample_id = int(perfect_indices[0])
    print(f"-> Đã chọn mẫu ID {teacher_sample_id + 1} (Điểm: 50.0/50.0) làm bài tập mẫu.")
else:
    teacher_sample_id = int(np.argmax(scores))
    print(f"-> Đã chọn mẫu ID {teacher_sample_id + 1} (Điểm cao nhất: {scores[teacher_sample_id]}) làm bài tập mẫu.")

start_row = teacher_sample_id * frames_per_sample
end_row = start_row + frames_per_sample
teacher_raw_data = df_x.iloc[start_row:end_row].values

teacher_rows = []
for frame_idx in range(frames_per_sample):
    row_x = teacher_raw_data[frame_idx]
    timestamp = round(frame_idx / FPS, 4)
    
    for joint_name, kinect_idx in KINECT_TO_MEDIAPIPE.items():
        base_col = kinect_idx * 4
        x = float(row_x[base_col])
        y = float(row_x[base_col + 1])
        z = float(row_x[base_col + 2])
        state = float(row_x[base_col + 3])
        
        visibility = 1.0 if state > 0 else 0.0
        if visibility == 0.0:
            x, y, z = np.nan, np.nan, np.nan
            
        teacher_rows.append({
            'Frame': frame_idx,
            'Timestamp': timestamp,
            'Joint_Name': joint_name,
            'X': x,
            'Y': y,
            'Z': z,
            'Visibility': visibility,
            'Label': 1  # 100% PASS
        })

df_teacher = pd.DataFrame(teacher_rows)
df_teacher.to_csv(OUTPUT_TEACHER_PATH, index=False)
print(f"-> Đã lưu file giáo viên: {OUTPUT_TEACHER_PATH} ({len(df_teacher):,} dòng, {frames_per_sample} frames, 100% Label 1)")

## 3. Kiểm tra tính hợp lệ của hai tệp vừa tạo

In [ ]:
print("=== 3. KIỂM TRA MẪU ĐẦU RA ===")
print("\n--- 5 dòng đầu student_dance_kimore.csv ---")
print(pd.read_csv(OUTPUT_STUDENT_PATH).head())

print("\n--- 5 dòng đầu teacher_dance_kimore.csv ---")
print(pd.read_csv(OUTPUT_TEACHER_PATH).head())
print("\n-> Dữ liệu đã sẵn sàng tích hợp trực tiếp vào pipeline trích xuất đặc trưng FastDTW & CatBoost!")